# Embed MCC + VSIC → artifact (.npz) on Google Colab

Notebook này chạy **Module 1 (`embed`)**: nhúng toàn bộ MCC + VSIC bằng `bge-m3` trên GPU của Colab và ghi ra một file artifact `embed-artifact.npz` lên Google Drive.

Sau khi chạy xong, tải `embed-artifact.npz` về máy local (hoặc dùng trực tiếp trên Drive) rồi chạy `map-vsic-mcc` ở bước 2 (notebook `mapping_vsic_mcc_colab.ipynb` hoặc local).

> **Lưu ý:** Bước này chỉ cần model embedding `bge-m3` — KHÔNG cần LLM.

## 1. Kết nối Google Drive
Artifact sẽ được ghi trực tiếp lên Google Drive để tránh mất dữ liệu khi runtime bị reset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Cài đặt Ollama
Cài Ollama để chạy model embedding local trên GPU của Colab.

In [ ]:
!apt-get update -y
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

## 3. Khởi động Ollama Server
Chạy Ollama server trong background.

In [ ]:
import os
import threading
import time

def run_ollama():
    os.system("ollama serve")

threading.Thread(target=run_ollama, daemon=True).start()
time.sleep(5)  # Đợi server khởi động

## 4. Tải Mô hình Embedding
Chỉ cần `bge-m3` (1024-dim) cho việc embedding.

In [ ]:
!ollama pull bge-m3

## 5. Clone code và cài đặt Dependencies
Clone repository về Google Drive, sau đó cài các thư viện **tối thiểu**.

> **Lưu ý:** Dùng `colab/requirements-mapping.txt` (đã đủ `numpy` + `ollama` cho bước embed) thay vì `requirements.txt` đầy đủ để tránh xung đột dependency và restart runtime.

In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/projects/mcc-lens"
REPO_URL = "https://github.com/hatuan314/mcc-lens.git"

os.makedirs(os.path.dirname(PROJECT_DIR), exist_ok=True)

if os.path.isdir(os.path.join(PROJECT_DIR, ".git")):
    os.chdir(PROJECT_DIR)
    !git pull
else:
    !git clone {REPO_URL} {PROJECT_DIR}
    os.chdir(PROJECT_DIR)

# Surya OCR chỉ cần cho feature image-to-JSON; gỡ để tránh xung đột httpx.
!pip uninstall -y surya-ocr || true

!pip install -q -r colab/requirements-mapping.txt

## 6. Chạy bước Embed
Dùng flag `--gdrive-output-dir` để ghi `embed-artifact.npz` vào thư mục trên Drive.

Đầu vào mặc định: `output/mcc-visa.json` + `output/vsic-vn.json` (đảm bảo chúng đã có trong repo/Drive).

In [ ]:
!python3 main.py embed \
  --mcc-input output/mcc-visa.json \
  --vsic-input output/vsic-vn.json \
  --gdrive-output-dir /content/drive/MyDrive/projects/mcc-lens \
  --embedding-model bge-m3

## 7. Kiểm tra artifact
Xác nhận file `embed-artifact.npz` đã được ghi và xem nhanh shape + meta.

In [ ]:
import json
import numpy as np

ARTIFACT = "/content/drive/MyDrive/projects/mcc-lens/embed-artifact.npz"
data = np.load(ARTIFACT, allow_pickle=True)
print("mcc_vectors:", data["mcc_vectors"].shape)
print("vsic_vectors:", data["vsic_vectors"].shape)
print("meta:", json.loads(str(data["meta"])))